# SemiconPlus Source-to-Target Contract

## Shared conventions

- Historical coverage: 2021-01-01 through 2025-12-31
- Stored timestamps: UTC
- Reporting timezone: Asia/Taipei
- Production-day boundary: 08:00 local time
- Synthetic records must contain a simulation flag and reproducible seed.
- Power BI uses conformed dimensions with no direct fact-to-fact relationships.

## Target inventory

| Target | Grain | Primary source | Classification |
|---|---|---|---|
| gold.dim_date | One row per date | Generated date range | DERIVED |
| gold.dim_site | One row per site | silver.sites | SOURCE |
| gold.dim_product_group | One row per product group | silver.product_groups | SOURCE |
| gold.dim_device | One row per device | silver.devices | SOURCE |
| gold.dim_equipment | One row per equipment | silver.equipment | SOURCE |
| gold.dim_lot | One row per source lot | Production lots and unit tests | SOURCE + DERIVED + MAPPED |
| gold.fact_yield_periodic | Date x site x product group x device | silver.production_lots | SOURCE + DERIVED |
| gold.fact_lot_performance | One row per source lot | Production lots and simulated retest | SOURCE + SYNTHETIC + DERIVED |
| gold.fact_retest_equipment | Hour x lot x equipment x defect/error | Simulated retest source | SYNTHETIC + DERIVED |
| gold.fact_oee_hourly | Production hour x equipment | Simulated hourly operations | SYNTHETIC + DERIVED |
| gold.fact_equipment_events | One row per observed event | silver.equipment_events | SOURCE + DERIVED |

## Production quantities

| Target field | Source or calculation | Classification |
|---|---|---|
| input_quantity | production_lots.quantity_started | SOURCE |
| first_pass_good_quantity | production_lots.quantity_passed | SOURCE |
| first_pass_fail_quantity | production_lots.quantity_failed | SOURCE |
| FPY | SUM(first_pass_good) / SUM(input) | DERIVED |
| retest_input_quantity | Simulated retest source | SYNTHETIC |
| retest_good_quantity | Simulated retest source | SYNTHETIC |
| retest_fail_quantity | Retest input - retest good | DERIVED |
| final_good_quantity | First-pass good + retest good | DERIVED |
| FTY | SUM(final_good) / SUM(input) | DERIVED |
| recovery_contribution | FTY - FPY | DERIVED |
| RPR | Retest good / retest input | DERIVED |

## Test-batch mapping

| Target field | Source or calculation | Classification |
|---|---|---|
| source_lot_id | production_lots.lot_id | SOURCE |
| lot_completion_timestamp_utc | Maximum unit-test timestamp; lot-start fallback | DERIVED |
| production_date | Asia/Taipei completion time using 08:00 boundary | DERIVED |
| test_batch_id | YY + device_id + MMDD | MAPPED |
| test_lot_sequence | Persistent sequence by completion time and source lot | MAPPED |
| test_lot_id | test_batch_id + T### | MAPPED |
| simulated_hierarchy_flag | true | DERIVED |

The test-batch hierarchy is a simulated analytical grouping. It is not physical
mother-lot, sublot, wafer-batch, or material genealogy.

## OEE mapping

| Target field | Source or calculation | Classification |
|---|---|---|
| scheduled_time_seconds | Simulated hourly operations | SYNTHETIC |
| approved_planned_downtime_seconds | Simulated hourly operations | SYNTHETIC |
| unplanned_downtime_seconds | Simulated hourly operations | SYNTHETIC |
| total_units | Simulated hourly operations | SYNTHETIC |
| good_units | Simulated hourly operations | SYNTHETIC |
| rated_units_per_hour | silver.equipment | SOURCE |
| planned_production_time_seconds | Scheduled - planned downtime | DERIVED |
| operating_time_seconds | Planned production - unplanned downtime | DERIVED |
| Availability | Operating / planned production | DERIVED |
| theoretical_output | Operating time x rated UPH / 3600 | DERIVED |
| Utilization | Total units / theoretical output | DERIVED |
| Quality | Good units / total units | DERIVED |
| OEE | Availability x Utilization x Quality | DERIVED |

## Known exclusions

- `silver.tester_logs` is excluded because it currently contains zero rows.
- Unit-test samples are not interpreted as retest attempts.
- Test-batch identifiers are not physical manufacturing genealogy.
- Observed equipment events are not used alone to calculate standard OEE.